# 🧠 Week 8 Lab — Student Version
## KNN: When Proximity Is the Algorithm

Last week we learned Naive Bayes: a generative model that builds class-conditional distributions. This week we take a fundamentally different approach — **K-Nearest Neighbors stores every training trial and classifies new trials by majority vote among the closest examples.** No parameters, no training phase, no assumptions about data distributions.

This lab follows the 7 sections of the Week 8 lecture, reproducing each of the 15 figures and building intuition about why KNN works, when it fails, and what the choice of distance metric reveals about your data.

**By the end of this lab, you will:**
- Implement KNN from scratch and with sklearn
- Reproduce the 100% EMG binary result and understand why it works
- Show that the population vector decoder is a special case of KNN
- Demonstrate why Mahalanobis distance fails in high dimensions
- Explore the curse of dimensionality and see PCA as regularization

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.spatial.distance import cosine, cdist

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneGroupOut
from sklearn.metrics import roc_curve, auc, confusion_matrix

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload week8_data.pkl

In [ ]:
# Load the dataset
with open('week8_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Unpack
X_raw = data['X_raw']            # (480, 6) EMG features
neural_rates = data['neural_rates']  # (480, 80) neural firing rates
targets = data['targets']         # 8 reach directions (0-7)
labels = data['labels']           # 'healthy' or 'impaired'
subjects = data['subjects']       # subject IDs (1-20)
muscle_names = data['muscle_names']
target_angles = data['target_angles']
dir_degrees = np.array([int(np.degrees(a)) for a in target_angles])
joint_angles = data['joint_angles']    # (480, 2) shoulder & elbow
joint_names = data['joint_names']
X_moons = data['X_moons']         # (300, 2) simulated clinical data
y_moons = data['y_moons']
moons_features = data['moons_feature_names']

group_binary = (labels == 'impaired').astype(int)
logo = LeaveOneGroupOut()

print(f"EMG: {X_raw.shape}, Neural: {neural_rates.shape}")
print(f"Subjects: {len(np.unique(subjects))}, Directions: {len(np.unique(targets))}")
print(f"Moons data: {X_moons.shape}")
print(f"Joint angles: {joint_angles.shape}")

---

## 🟢 Part 1: KNN from Scratch — The Algorithm (Lecture §1)

The lecture introduced KNN as the simplest possible classifier: store everything, then vote. Before using sklearn, let's implement it by hand to understand exactly what happens at each step.

### Exercise 1.1: KNN decision boundaries on clinical data (Lecture Figure 1)

**Learning objective:** See how k controls the bias-variance tradeoff visually.

Using the simulated clinical screening data (`X_moons`, `y_moons`), fit KNN at three k values (1, 11, 201) and plot decision boundaries. Use `meshgrid` to create a grid, predict on each grid point, and use `contourf` to shade regions.

*Hint:* Create the grid with `np.meshgrid(np.linspace(...), np.linspace(...))`, reshape for prediction, then reshape back.

In [ ]:
# Exercise 1.1: KNN decision boundaries at k=1, 11, 201
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, k, title in [(axes[0], 1, 'k=1 (Overfit)'),
                      (axes[1], 11, 'k=11 (Balanced)'),
                      (axes[2], 201, 'k=201 (Oversmooth)')]:
    
    # TODO: Fit KNN on X_moons, y_moons
    # knn = ...
    
    # TODO: Create meshgrid for decision boundary
    # h = 0.05  # step size
    # xx, yy = np.meshgrid(...)
    # Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # TODO: Plot with contourf, scatter the data points, label axes
    # ax.contourf(xx, yy, Z, ...)
    # ax.scatter(...)
    
    pass

plt.suptitle('KNN Decision Boundaries: How k Controls Complexity', fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 1.2: Implement KNN from scratch

**Learning objective:** Understand every step of the algorithm — distance computation, sorting, voting.

Write a function that classifies a single test point using KNN with Euclidean distance. Then test it on the moons data.

In [ ]:
# Exercise 1.2: KNN from scratch
def knn_predict_single(X_train, y_train, x_test, k=5):
    """
    Classify a single test point using KNN.
    
    Steps:
    1. Compute Euclidean distance from x_test to every training point
    2. Find the k nearest neighbors (argsort)
    3. Return the majority vote among those k labels
    """
    # TODO: Compute distances
    # dists = ...
    
    # TODO: Find k nearest indices
    # nn_idx = ...
    
    # TODO: Majority vote
    # nn_labels = y_train[nn_idx]
    # prediction = ...
    
    return None  # Replace with your prediction

# Test: classify the first 10 moons points using the rest as training
for i in range(10):
    mask = np.ones(len(X_moons), dtype=bool)
    mask[i] = False
    pred = knn_predict_single(X_moons[mask], y_moons[mask], X_moons[i], k=5)
    print(f"Point {i}: true={y_moons[i]}, predicted={pred}")

---

## 🟢 Part 2: KNN on All Four Tasks (Lecture §2)

Now we move from the toy clinical data to our real motor control dataset. The lecture showed KNN achieves 100% on EMG binary — perfect subject-level clustering — and 94.6% on neural direction decoding.

### Exercise 2.1: PCA visualization of neural data (Lecture Figure 2)

**Learning objective:** See the low-dimensional structure that makes KNN work.

Apply PCA to the 80-neuron data. Plot trials in PC1–PC2 space colored by direction, and show the variance explained bar chart.

In [ ]:
# Exercise 2.1: PCA visualization
sc = StandardScaler()
pca = PCA()
X_pca = pca.fit_transform(sc.fit_transform(neural_rates))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) Scatter by direction
cmap = plt.cm.hsv(np.linspace(0, 1, 8, endpoint=False))
for d in range(8):
    mask = targets == d
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[cmap[d]], s=15, alpha=0.5,
                   label=f'{dir_degrees[d]}°')
# TODO: Add labels, legend, title

# (b) Variance explained
# TODO: Bar chart of pca.explained_variance_ratio_[:20]

plt.tight_layout()
plt.show()

print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}% variance")
print(f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}% variance")
print(f"PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}% variance")

### Exercise 2.2: KNN boundaries in PCA space (Lecture Figure 3)

**Learning objective:** See how KNN partitions the neural direction space at different k values.

In [ ]:
# Exercise 2.2: KNN boundaries on neural PCA(2) data
X_neur_pca2 = X_pca[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmap8 = plt.cm.hsv(np.linspace(0, 1, 8, endpoint=False))

for ax, k in zip(axes, [1, 5, 101]):
    # TODO: Fit KNN on X_neur_pca2, targets
    # TODO: Create meshgrid and plot decision boundaries
    # TODO: Scatter training points colored by direction
    
    ax.set_title(f'k={k}', fontweight='bold')

plt.suptitle('KNN Decision Boundaries in Neural PCA Space', fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 2.3: LOSO cross-validation — all four tasks (Lecture Figure 4)

**Learning objective:** Run the complete evaluation and compare with LR and NB from previous weeks.

Run KNN (k=5, Euclidean) with LOSO cross-validation on all four task/feature combinations. Plot alongside LR and NB results.

In [ ]:
# Exercise 2.3: LOSO for all four tasks
pipe_knn = Pipeline([('s', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=5))])

# TODO: Run cross_val_score for each combination:
# 1. EMG direction: pipe_knn on X_raw, targets
# 2. EMG binary: pipe_knn on X_raw, group_binary
# 3. Neural direction: pipe_knn on neural_rates, targets  
# 4. Neural binary: pipe_knn on neural_rates, group_binary

# Also run LR and NB for comparison
pipe_lr = Pipeline([('s', StandardScaler()),
                    ('lr', LogisticRegression(C=10, max_iter=5000, solver='lbfgs'))])
pipe_nb = Pipeline([('s', StandardScaler()),
                    ('nb', GaussianNB())])

# knn_emg_dir = cross_val_score(...).mean() * 100
# ... (compute all 12 accuracy values)

# TODO: Create grouped bar chart comparing KNN, LR, NB across all 4 tasks


### Exercise 2.4: Why 100%? — Subject cluster visualization (Lecture Figure 5)

**Learning objective:** Understand that KNN's perfect EMG binary accuracy reflects clean subject-level separation, not overfitting.

Compute the centroid of each subject's 24 trials in EMG PCA space. Color by group. Show that each subject's nearest centroid neighbor is always from the correct group.

In [ ]:
# Exercise 2.4: Subject clusters in EMG PCA space
sc_emg = StandardScaler()
pca_emg = PCA(n_components=2)
X_emg_pca = pca_emg.fit_transform(sc_emg.fit_transform(X_raw))

# Compute per-subject centroids
unique_subjs = np.unique(subjects)
centroids = np.array([X_emg_pca[subjects == s].mean(axis=0) for s in unique_subjs])
subj_labels = np.array([group_binary[subjects == s][0] for s in unique_subjs])

# TODO: Plot centroids colored by group (healthy=blue, impaired=red)
# TODO: Draw lines from each subject's trials to their centroid
# TODO: For one test subject, draw lines to its k nearest centroid neighbors
# TODO: Show that the nearest neighbors are always from the correct group

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# (a) All subjects with trial clouds
# (b) KNN on centroids — show nearest neighbors for one test subject

plt.tight_layout()
plt.show()

### Exercise 2.5: Joint angles — biomechanics reveals the task (Lecture Figure 6)

**Learning objective:** Understand why joint angles achieve 99.2% on direction but below chance on binary diagnosis.

Run KNN on joint angles for both tasks and compare with EMG and neural features.

In [ ]:
# Exercise 2.5: Three feature sets compared
feature_sets = {
    'EMG (6)': X_raw,
    'Neural (80)': neural_rates,
    'Joint Angles (2)': joint_angles,
}

results = {}
for name, X in feature_sets.items():
    pipe = Pipeline([('s', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=5))])
    results[f'{name}_dir'] = cross_val_score(pipe, X, targets, cv=logo, groups=subjects).mean() * 100
    results[f'{name}_bin'] = cross_val_score(pipe, X, group_binary, cv=logo, groups=subjects).mean() * 100

# TODO: Create a grouped bar chart: 3 feature sets x 2 tasks
# TODO: Add a chance line for each task (12.5% for direction, 50% for binary)
# TODO: Annotate the key finding: joint angles 99.2% direction, 42.9% binary

for name in feature_sets:
    print(f"{name}: direction={results[f'{name}_dir']:.1f}%, binary={results[f'{name}_bin']:.1f}%")

### 🤔 Thought Exercise
Why do joint angles score 99.2% on direction but only 42.9% on binary? Think about what information joint angles encode versus what group membership depends on.

In [ ]:
# Your reflection:


---

## 🟡 Part 3: Distance Metrics — What "Close" Means (Lecture §3)

The lecture showed that the choice of distance metric encodes your domain knowledge. Cosine distance outperforms Euclidean on neural direction decoding because it measures the *direction* of activity patterns, not their magnitude.

### Exercise 3.1: Compare distance metrics (Lecture Figure 7a)

**Learning objective:** See that the metric is a modeling decision, not a technical detail.

In [ ]:
# Exercise 3.1: Distance metric comparison
metrics = ['euclidean', 'manhattan', 'cosine']
metric_names = ['Euclidean', 'Manhattan', 'Cosine']

neur_accs = []
emg_accs = []
for m in metrics:
    pipe = Pipeline([('s', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=5, metric=m))])
    neur_accs.append(cross_val_score(pipe, neural_rates, targets, cv=logo, groups=subjects).mean() * 100)
    emg_accs.append(cross_val_score(pipe, X_raw, targets, cv=logo, groups=subjects).mean() * 100)

# TODO: Create grouped bar chart of EMG vs Neural accuracy by metric
# TODO: Add PopVec dashed line at 95.6%
# TODO: Which metric works best for neural? For EMG? Why?

for name, n, e in zip(metric_names, neur_accs, emg_accs):
    print(f"{name}: Neural={n:.1f}%, EMG={e:.1f}%")

### Exercise 3.2: Visualize cosine vs Euclidean (Lecture Figure 7b)

**Learning objective:** Build geometric intuition — cosine measures direction, Euclidean measures amplitude + direction.

Create three vectors from the origin: Trial A and Trial B point the same direction (different amplitudes), Trial C points a different direction. Compute both distances.

In [ ]:
# Exercise 3.2: Vector diagram — cosine vs Euclidean
from scipy.spatial.distance import cosine as cos_dist

# Three vectors from origin
angle_AB = np.radians(30)   # Trials A and B: same direction (30°)
angle_C = np.radians(75)    # Trial C: different direction (75°)

vA = np.array([1.0 * np.cos(angle_AB), 1.0 * np.sin(angle_AB)])   # moderate amplitude
vB = np.array([1.8 * np.cos(angle_AB), 1.8 * np.sin(angle_AB)])   # higher amplitude, same direction
vC = np.array([1.1 * np.cos(angle_C), 1.1 * np.sin(angle_C)])     # similar amplitude, different direction

# TODO: Compute cosine and Euclidean distances for A-vs-B and A-vs-C
# cos_AB = cos_dist(vA, vB)
# euc_AB = np.linalg.norm(vA - vB)
# cos_AC = cos_dist(vA, vC)
# euc_AC = np.linalg.norm(vA - vC)

# TODO: Plot the three vectors as arrows from origin
# TODO: Draw angle arc between red and blue directions
# TODO: Add text boxes showing both distances for each pair

# Key insight: A vs B have cosine ≈ 0 (same direction!) but Euclidean ≈ 0.8 (different amplitudes)
#              A vs C have cosine ≈ 0.29 (different directions) and Euclidean ≈ 0.81


### Exercise 3.3: PopVec is KNN with cosine distance and k=all (Lecture Figure 8)

**Learning objective:** This is the key theoretical insight of the lecture. Demonstrate empirically that the population vector decoder and KNN with cosine distance (k=all training trials) make identical predictions.

**Steps:**
1. Compute PopVec predictions via LOSO: in each fold, compute direction means from training data, classify test trials by smallest cosine angle to the means
2. Compute KNN cosine (k=all) predictions via LOSO
3. Compare trial-by-trial

In [ ]:
# Exercise 3.3a: Population Vector LOSO predictions
popvec_preds = np.zeros(len(targets), dtype=int)

for train_idx, test_idx in logo.split(neural_rates, targets, subjects):
    X_tr, y_tr = neural_rates[train_idx], targets[train_idx]
    X_te = neural_rates[test_idx]
    
    # TODO: Compute direction means from training data
    # dir_means = np.array([X_tr[y_tr == d].mean(0) for d in range(8)])
    
    # TODO: For each test trial, find the direction with highest cosine similarity
    # for i, x in enumerate(X_te):
    #     sims = [1 - cosine(x, dir_means[d]) for d in range(8)]
    #     popvec_preds[test_idx[i]] = np.argmax(sims)

popvec_acc = (popvec_preds == targets).mean() * 100
print(f"PopVec LOSO accuracy: {popvec_acc:.1f}%")

In [ ]:
# Exercise 3.3b: KNN cosine (k=all) LOSO predictions
knn_cos_all_preds = np.zeros(len(targets), dtype=int)

for train_idx, test_idx in logo.split(neural_rates, targets, subjects):
    X_tr, y_tr = neural_rates[train_idx], targets[train_idx]
    X_te = neural_rates[test_idx]
    
    # TODO: For each test trial, sum cosine similarities to all training trials per direction
    # For each direction d: sum_sim_d = sum of cos_sim(x_test, x_i) for all x_i in direction d
    # Predict direction with highest total similarity
    
    # KNN with k=all and distance weighting is equivalent to summing similarities per class
    
    pass

agreement = (popvec_preds == knn_cos_all_preds).sum()
print(f"Agreement: {agreement}/{len(targets)} trials ({agreement/len(targets)*100:.1f}%)")

In [ ]:
# Exercise 3.3c: Similarity profile for one trial (Lecture Figure 8b)
# Pick one test trial and show the similarity profile for both methods

# TODO: For a single fold, pick one test trial
# TODO: Compute cos_sim(trial, mean_d) for all 8 directions — this is PopVec
# TODO: Compute avg cos_sim(trial, each training trial in direction d) — this is KNN
# TODO: Bar chart comparing the two profiles side by side



In [ ]:
# Exercise 3.3d: k-sweep convergence (Lecture Figure 8d)
# Show that as k increases, KNN cosine predictions converge to PopVec predictions

k_values = [1, 3, 5, 9, 15, 21, 51, 101, 201, 456]

# TODO: For each k, run KNN cosine LOSO, compare predictions to PopVec
# TODO: Average agreement across all 20 folds
# TODO: Plot agreement vs k — should rise from ~95% at k=1 to 100% at k=all



---

## 🟡 Part 4: PCA + Euclidean vs Mahalanobis (Lecture §3, Figure 9)

The lecture showed that PCA + Euclidean and Mahalanobis distance are related but **not equivalent**. Mahalanobis weights each PC by 1/eigenvalue, which amplifies noise in high-dimensional data. This is one of the most important practical lessons about distance metrics.

### Exercise 4.1: Eigenvalue spectrum (Lecture Figure 9a)

**Learning objective:** See that most variance is in PCs 1–2 (direction signal) and the rest is noise.

In [ ]:
# Exercise 4.1: Eigenvalue spectrum
sc = StandardScaler()
pca_all = PCA()
X_std = pca_all.fit_transform(sc.fit_transform(neural_rates))

# TODO: Bar chart of all 80 eigenvalues
# TODO: Highlight PCs 1-2 (signal) vs PCs 3-80 (noise)

eigenvalues = pca_all.explained_variance_
print(f"PC1 eigenvalue: {eigenvalues[0]:.1f}")
print(f"PC2 eigenvalue: {eigenvalues[1]:.1f}")
print(f"PC80 eigenvalue: {eigenvalues[-1]:.4f}")
print(f"Ratio PC1/PC80: {eigenvalues[0]/eigenvalues[-1]:.0f}x")

### Exercise 4.2: Mahalanobis collapses with more PCs (Lecture Figure 9c)

**Learning objective:** See that Mahalanobis accuracy drops steadily as you include more PCs, while PCA + Euclidean stays robust.

For each number of PCA components (2, 5, 10, 20, 40, 80):
- Run PCA(n) + Euclidean KNN with LOSO
- Run Mahalanobis KNN in PCA(n) space with LOSO (use `scipy.spatial.distance.cdist` with `metric='mahalanobis'`)

In [ ]:
# Exercise 4.2: PCA+Euclidean vs Mahalanobis at different truncation levels
from scipy.spatial.distance import cdist

n_components_list = [2, 5, 10, 20, 40, 80]
pca_euc_accs = []
mahal_accs = []

for n_comp in n_components_list:
    # PCA + Euclidean (easy — use Pipeline)
    pipe = Pipeline([('s', StandardScaler()),
                     ('pca', PCA(n_components=n_comp)),
                     ('knn', KNeighborsClassifier(n_neighbors=5))])
    pca_acc = cross_val_score(pipe, neural_rates, targets, cv=logo, groups=subjects).mean() * 100
    pca_euc_accs.append(pca_acc)
    
    # TODO: Mahalanobis in PCA(n) space — manual implementation needed
    # For each LOSO fold:
    #   1. StandardScaler + PCA(n_comp) on training data
    #   2. Transform test data
    #   3. Compute covariance of training data in PCA space
    #   4. Add small regularization: cov += np.eye(n_comp) * 1e-6
    #   5. Invert covariance
    #   6. Use cdist(X_te, X_tr, metric='mahalanobis', VI=cov_inv)
    #   7. Find k=5 nearest, majority vote
    
    mahal_accs.append(0)  # Replace with your computation
    
    print(f"n_comp={n_comp:3d}: PCA+Euc={pca_acc:.1f}%, Mahal=___")

# TODO: Grouped bar chart comparing the two methods across component counts


### 🤔 Thought Exercise
Why does Mahalanobis accuracy *decrease* as you add more PCA components? Think about what dividing by a tiny eigenvalue does to the distance computation.

In [ ]:
# Your reflection:


---

## 🔴 Part 5: The Curse of Dimensionality (Lecture §4)

The lecture demonstrated that KNN degrades dramatically as irrelevant features are added. Unlike parametric models that can learn to ignore noise features, KNN treats every dimension equally in its distance computation.

### Exercise 5.1: Adding noise features destroys KNN (Lecture Figure 10)

**Learning objective:** See the curse of dimensionality in action.

Start with 6 EMG features. Progressively add random noise features (10, 50, 100, 500, 1000) and measure KNN accuracy. Compare with LR and NB.

In [ ]:
# Exercise 5.1: Curse of dimensionality
np.random.seed(42)
noise_counts = [0, 10, 50, 100, 500, 1000]
knn_accs = []
lr_accs = []
nb_accs = []

for n_noise in noise_counts:
    # Add random noise features to EMG
    if n_noise > 0:
        noise = np.random.randn(len(X_raw), n_noise)
        X_aug = np.hstack([X_raw, noise])
    else:
        X_aug = X_raw.copy()
    
    # TODO: Run LOSO for KNN, LR, NB on X_aug for direction decoding
    # knn_acc = cross_val_score(pipe_knn, X_aug, targets, cv=logo, groups=subjects).mean() * 100
    # lr_acc = ...
    # nb_acc = ...
    
    pass

# TODO: Line plot of accuracy vs number of noise features for all 3 methods
# TODO: Annotate: KNN drops to near-chance, LR and NB are more robust


### Exercise 5.2: PCA rescues KNN (Lecture Figure 11)

**Learning objective:** Dimensionality reduction before KNN prevents the curse.

Sweep the number of PCA components from 2 to 80 and measure KNN LOSO accuracy on neural direction and neural binary.

In [ ]:
# Exercise 5.2: PCA component sweep for KNN
n_comps = [2, 5, 10, 20, 30, 50, 80]
dir_accs = []
bin_accs = []

for n in n_comps:
    pipe = Pipeline([('s', StandardScaler()),
                     ('pca', PCA(n_components=n)),
                     ('knn', KNeighborsClassifier(n_neighbors=5))])
    
    # TODO: Run LOSO for direction and binary tasks
    # dir_accs.append(...)
    # bin_accs.append(...)
    pass

# TODO: Plot accuracy vs n_components for both tasks
# TODO: Add horizontal lines for raw KNN accuracy (no PCA)
# Key finding: PCA(2) achieves 95.4% on direction — better than raw KNN at 94.6%


---

## 🔴 Part 6: Choosing k and Distance Weighting (Lecture §5)

The lecture swept k from 1 to 31 and found that the optimal k depends on the task. It also showed that distance weighting has no effect on our well-separated data, but matters when classes overlap.

### Exercise 6.1: k sweep across all four tasks (Lecture Figure 12)

**Learning objective:** See the bias-variance tradeoff in action — small k overfits, large k oversmooths.

In [ ]:
# Exercise 6.1: k sweep
k_values = list(range(1, 32, 2))
tasks = [
    ('EMG Direction', X_raw, targets, 12.5),
    ('EMG Binary', X_raw, group_binary, 50),
    ('Neural Direction', neural_rates, targets, 12.5),
    ('Neural Binary', neural_rates, group_binary, 50),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (title, X, y, chance) in zip(axes, tasks):
    accs = []
    for k in k_values:
        pipe = Pipeline([('s', StandardScaler()),
                         ('knn', KNeighborsClassifier(n_neighbors=k))])
        # TODO: Run LOSO, append accuracy
        pass
    
    # TODO: Plot accuracy vs k, add chance line, highlight best k
    ax.set_title(title, fontweight='bold')

plt.suptitle('Choosing k: The Bias-Variance Tradeoff', fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 6.2: When does distance weighting matter? (Lecture Figure 13)

**Learning objective:** Understand why weighting has no effect on well-separated data but changes results when classes overlap.

Test uniform vs distance-weighted KNN on both our real data and the overlapping clinical simulation.

In [ ]:
# Exercise 6.2a: Uniform vs weighted on our data
# TODO: For neural direction at k=11, compare uniform and weighted KNN
pipe_uniform = Pipeline([('s', StandardScaler()),
                         ('knn', KNeighborsClassifier(n_neighbors=11, weights='uniform'))])
pipe_weighted = Pipeline([('s', StandardScaler()),
                          ('knn', KNeighborsClassifier(n_neighbors=11, weights='distance'))])

# acc_u = cross_val_score(pipe_uniform, neural_rates, targets, cv=logo, groups=subjects).mean() * 100
# acc_w = cross_val_score(pipe_weighted, neural_rates, targets, cv=logo, groups=subjects).mean() * 100
# print(f"Uniform: {acc_u:.1f}%, Weighted: {acc_w:.1f}%")
# They should be identical or nearly so — why?


In [ ]:
# Exercise 6.2b: Create overlapping data where weighting matters
np.random.seed(42)
n_per_class = 150
xA = np.random.randn(n_per_class, 2) * 0.30 + [0.7, 0.75]  # Mild impairment
xB = np.random.randn(n_per_class, 2) * 0.30 + [0.4, 0.45]  # Moderate impairment
X_overlap = np.vstack([xA, xB])
y_overlap = np.array([0]*n_per_class + [1]*n_per_class)

# TODO: Fit KNN with k=11 using both uniform and distance weighting
# TODO: Plot decision boundaries for both on the same figure
# TODO: Find a test point where the two methods disagree
# Hint: look for points where the k neighbors are a mix of both classes


---

## 🔴 Part 7: Computational Cost and Summary (Lecture §6–7)

KNN's computational profile is the opposite of parametric models: training is instant (store data), but prediction is slow (compare to every training point).

### Exercise 7.1: Timing comparison (Lecture Figure 14)

**Learning objective:** Measure KNN's train-predict asymmetry and compare with LR.

In [ ]:
# Exercise 7.1: Timing comparison
import time

models = {
    'KNN (k=5)': Pipeline([('s', StandardScaler()),
                            ('knn', KNeighborsClassifier(n_neighbors=5))]),
    'Logistic Reg': Pipeline([('s', StandardScaler()),
                               ('lr', LogisticRegression(C=10, max_iter=5000))]),
    'Naive Bayes': Pipeline([('s', StandardScaler()),
                              ('nb', GaussianNB())]),
}

# Time one LOSO fold for each model
train_idx, test_idx = next(logo.split(neural_rates, targets, subjects))
X_tr, y_tr = neural_rates[train_idx], targets[train_idx]
X_te = neural_rates[test_idx]

for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    t_train = time.time() - t0
    
    t0 = time.time()
    for _ in range(100):  # repeat for measurable time
        pipe.predict(X_te)
    t_pred = (time.time() - t0) / 100
    
    print(f"{name:15s}: train={t_train*1000:.2f}ms, predict={t_pred*1000:.4f}ms")

### Exercise 7.2: The comparison table (Lecture Figure 15)

**Learning objective:** Compile all results into the running comparison table.

In [ ]:
# Exercise 7.2: Compile all results
print("Running Comparison Table Through Week 8")
print("=" * 70)
print(f"{'Week':>4} {'Method':>20} {'Features':>14} {'8-Dir':>7} {'Binary':>7}")
print("-" * 70)
# Previous weeks (from lecture):
print(f"{'5':>4} {'LR (C=10)':>20} {'Raw EMG (6)':>14} {'80.8%':>7} {'85.0%':>7}")
print(f"{'6':>4} {'LR (C=10)':>20} {'Neural (80)':>14} {'91.2%':>7} {'73.3%':>7}")
print(f"{'6':>4} {'Pop. Vector':>20} {'Neural (80)':>14} {'95.6%':>7} {'N/A':>7}")
print(f"{'7':>4} {'Gaussian NB':>20} {'Raw EMG (6)':>14} {'81.2%':>7} {'65.4%':>7}")
print(f"{'7':>4} {'Gaussian NB':>20} {'Neural (80)':>14} {'95.2%':>7} {'77.7%':>7}")
print(f"{'7':>4} {'PCA(6) + NB':>20} {'Raw EMG (6)':>14} {'—':>7} {'85.0%':>7}")
# TODO: Add Week 8 KNN results from your computations
# print(f"{'8':>4} {'KNN (k=5)':>20} {'Raw EMG (6)':>14} {knn_emg_dir:>6.1f}% {knn_emg_bin:>6.1f}%")
# print(f"{'8':>4} {'KNN (k=5)':>20} {'Neural (80)':>14} {knn_neur_dir:>6.1f}% {knn_neur_bin:>6.1f}%")
# print(f"{'8':>4} {'KNN (cosine)':>20} {'Neural (80)':>14} {knn_cos_dir:>6.1f}% {'—':>7}")


### 🤔 Final Thought Exercise

The lecture argued that the distance metric is where domain knowledge enters KNN. Cosine distance "knows" that neural direction matters but amplitude doesn't. Euclidean distance treats them equally.

**Question:** If you were building a clinical BCI (brain-computer interface) for a patient with ALS, which distance metric would you choose for decoding intended movement direction from motor cortex recordings? What if you also wanted to decode the *vigor* (speed/force) of the intended movement? Would you use the same metric for both tasks?

In [ ]:
# Your reflection:


---

## Summary

This lab followed the 7 sections of the Week 8 lecture:

1. **§1 (Part 1):** Implemented KNN from scratch and visualized decision boundaries
2. **§2 (Part 2):** Tested KNN on all four motor control tasks — 100% EMG binary, 94.6% neural direction
3. **§3 (Parts 3–4):** Explored distance metrics, proved PopVec = KNN cosine (k=all), and showed PCA+Euclidean ≠ Mahalanobis
4. **§4 (Part 5):** Demonstrated the curse of dimensionality and PCA as rescue
5. **§5 (Part 6):** Swept k and explored when distance weighting matters
6. **§6–7 (Part 7):** Measured computational costs and compiled the running comparison table

**Key takeaways:**
- KNN has no training phase — it stores everything and decides at prediction time
- The distance metric encodes your domain knowledge (cosine vs Euclidean vs Mahalanobis)
- The population vector is secretly KNN with cosine distance and k=all
- PCA before KNN is not just dimensionality reduction — it's regularization
- KNN achieves 100% on EMG binary because subjects form well-separated clusters